# Performance and Best Practices

Small tables never show performance problems. Real projects do: a 5 GB CSV, a feature step that takes 40 minutes, a laptop that runs out of memory. This chapter teaches how to make pandas code **use less memory, run faster, and stay readable**, and how to know when pandas is the wrong tool.

> **Rule 0: measure before you optimize.** A CSV file's size on disk is not a reliable estimate of pandas' RAM usage.
Depending on the dtypes, strings, intermediate results, indexes, and the operations
being performed, a DataFrame can use several times the file size in memory.

For example, a 2 GB CSV may require substantially more than 2 GB of RAM.
Measure the real DataFrame with `memory_usage(deep=True)` instead of relying on
a fixed multiplier.
**Contents**
1. The performance mindset
2. Measuring time and memory
3. Our dataset (500,000 transactions)
4. Memory: choosing the right dtypes
5. `category` in depth
6. Speed: vectorization (measured)
7. Filtering, `query`, `eval`, and doing less work
8. GroupBy and merge performance
9. File formats and reading less
10. Processing files bigger than memory (chunks)
11. Copies, Copy-on-Write and freeing memory
12. Method chaining and clean pipelines
13. When pandas is not enough: DuckDB and Polars
14. Mini AI/ML example: an efficient feature pipeline
15. Pandas 3.x notes
16. Common mistakes
17. Interview questions
18. Practice questions with solutions
19. Cheat sheet

In [2]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

DATA = Path("data")
DATA.mkdir(exist_ok=True)

pandas: 3.0.6


---
# 1. The performance mindset

Two different problems, two different fixes:

| Problem | Symptom | Main fixes |
|---|---|---|
| **Memory** | `MemoryError`, laptop freezes, swap | right dtypes, load fewer columns/rows, chunks, Parquet |
| **Speed** | steps take minutes | vectorize, avoid `apply`/loops, filter early, built-in aggregations |

The **order of importance** for most pandas code:

1. **Load less** (only the columns and rows you need)
2. **Right dtypes** (`category`, small ints, `datetime64`)
3. **Vectorize** (no Python loops over rows)
4. **Do less work** (filter early, select columns before merge/groupby)
5. **Use better tools** when data outgrows pandas (DuckDB, Polars, a database)

---
# 2. Measuring time and memory

## 2.1 Time

In [3]:
def bench(func, repeat=3):
    """Run func several times and return the BEST time in seconds (the best run has the least noise)."""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        func()
        best = min(best, time.perf_counter() - start)
    return best

s = pd.Series(np.random.default_rng(0).random(1_000_000))
print(f"sum of 1,000,000 numbers: {bench(lambda: s.sum()) * 1000:.2f} ms")

sum of 1,000,000 numbers: 0.78 ms


In [4]:
# In Jupyter you can also use the magic command (it repeats the run many times for you)
%timeit s.sum()

394 μs ± 21.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


- `%timeit expr`: one line. `%%timeit` at the top of a cell: the whole cell.
- Use **`time.perf_counter()`** for a single measurement and `%timeit` for small, fast operations.
- Timings change from machine to machine and from run to run. Compare **ratios** (A is 20x slower than B), not absolute numbers.

## 2.2 Memory

In [5]:
def mb(obj):
    """Memory of a DataFrame/Series in MB (deep=True counts the real size of text)."""
    return round(obj.memory_usage(deep=True).sum() / 1024 ** 2, 2)

demo = pd.DataFrame({"city": ["Indore", "Bhopal"] * 5, "n": range(10)})
print(demo.memory_usage(deep=True))           # bytes per column (+ the index)
print("total MB:", mb(demo))

Index    132
city     140
n         80
dtype: int64
total MB: 0.0


`df.info(memory_usage="deep")` shows the same in the summary. **`deep=True` matters** for text columns: without it pandas only counts the pointers, not the text.

---
# 3. Our dataset (500,000 transactions)

Big enough to feel real, small enough to run on any laptop.

In [6]:
rng = np.random.default_rng(42)
n = 500_000

tx = pd.DataFrame({
    "tx_id":       np.arange(n),
    "customer_id": rng.integers(1, 20_001, n),
    "city":        rng.choice(["Indore", "Bhopal", "Delhi", "Pune", "Mumbai"], n),
    "category":    rng.choice(["Grocery", "Fashion", "Electronics", "Home", "Travel", "Food"], n),
    "amount":      rng.lognormal(5, 1, n).round(2),
    "quantity":    rng.integers(1, 10, n),
    "is_online":   rng.random(n) < 0.4,
    "date":        pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
})

print(tx.shape)
tx.head()

(500000, 8)


,tx_id,customer_id,city,category,amount,quantity,is_online,date
0,0,1786,Mumbai,Travel,108.66,2,False,2025-02-12
1,1,15480,Mumbai,Grocery,104.86,1,False,2025-01-01
2,2,13092,Bhopal,Home,129.72,4,False,2025-09-22
3,3,8778,Pune,Food,173.40,4,True,2025-09-02
4,4,8661,Bhopal,Electronics,90.53,5,False,2025-08-03


In [7]:
tx.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   tx_id        500000 non-null  int64         
 1   customer_id  500000 non-null  int64         
 2   city         500000 non-null  str           
 3   category     500000 non-null  str           
 4   amount       500000 non-null  float64       
 5   quantity     500000 non-null  int64         
 6   is_online    500000 non-null  bool          
 7   date         500000 non-null  datetime64[us]
dtypes: bool(1), datetime64[us](1), float64(1), int64(3), str(2)
memory usage: 32.9 MB


In [8]:
before = tx.memory_usage(deep=True).drop("Index").rename("bytes").to_frame()
before["MB"] = (before["bytes"] / 1024 ** 2).round(2)
before["dtype"] = tx.dtypes.astype(str)
before.sort_values("bytes", ascending=False)

,bytes,MB,dtype
category,7250763,6.91,str
city,6700508,6.39,str
tx_id,4000000,3.81,int64
customer_id,4000000,3.81,int64
amount,4000000,3.81,float64
quantity,4000000,3.81,int64
date,4000000,3.81,datetime64[us]
is_online,500000,0.48,bool


Every column uses a **default, generous dtype**: `int64` (8 bytes) for `quantity` that only holds 1 to 9, `int64` for `customer_id`, text for 5 cities repeated 100,000 times, and a full `bool` as 1 byte. We can do much better.

# 4. Memory: choosing the right dtypes

## 4.1 Integer sizes

| dtype | bytes | range |
|---|---|---|
| `int8` | 1 | -128 to 127 |
| `int16` | 2 | -32,768 to 32,767 |
| `int32` | 4 | about -2.1 billion to 2.1 billion |
| `int64` | 8 | huge (default) |
| `uint8` | 1 | 0 to 255 (no negatives) |
| `float32` | 4 | about 7 significant digits |
| `float64` | 8 | about 15-16 significant digits (default) |

## 4.2 The danger: silent overflow

In [9]:
values = pd.Series([5, 120, 200])
print(values.astype("int8").tolist())          # 200 does not fit in int8...

[5, 120, -56]


**No error. `200` silently became `-56`.** This is the biggest risk of manual downcasting. Never use `astype("int8")` blindly.

The safe way for **integers**: **`pd.to_numeric(..., downcast="integer")`** picks the smallest integer type that really fits the data:

In [10]:
print(pd.to_numeric(values, downcast="integer").dtype)           # int16 (because of 200)
print(pd.to_numeric(tx["quantity"], downcast="integer").dtype)    # int8   (values 1 to 9)
print(pd.to_numeric(tx["customer_id"], downcast="integer").dtype) # int16  (values up to 20,000)
print(pd.to_numeric(tx["tx_id"], downcast="integer").dtype)       # int32  (values up to 500,000)

int16
int8
int16
int32


## 4.3 Floats: be careful with money

`float32` has only about 7 significant digits. For measurements (temperature, a model input) that is fine. For **money or IDs** a rounding error is a real error. Let us check what `float32` does to our amounts:

In [11]:
amount32 = tx["amount"].astype("float32")
max_error = (tx["amount"] - amount32).abs().max()
print("largest error after float32:", round(max_error, 6))
print("identical after conversion:", (tx["amount"] == amount32).all())

largest error after float32: 0.00043
identical after conversion: False


The values change a little (small, but real). Now the trap: `pd.to_numeric(downcast="float")` looks like the "safe" version, **but it is not lossless**.

In [12]:
auto32 = pd.to_numeric(tx["amount"], downcast="float")
print("dtype chosen:", auto32.dtype)
print("identical to the original:", (auto32 == tx["amount"]).all())

dtype chosen: float32
identical to the original: False


It converted to `float32` even though the values changed. For floats the downcast only checks the **range**, not the **precision**. So for floats you must check it yourself: convert only if the values survive a round trip **exactly**.

In [13]:
def float32_is_lossless(s):
    return bool((s.astype("float32").astype("float64") == s).all())

print("amounts (2 decimals):", float32_is_lossless(tx["amount"]))                       # False: keep float64
print("small whole numbers  :", float32_is_lossless(pd.Series([1.0, 2.0, 3.5, 100.25])))  # True: safe (exactly representable)

amounts (2 decimals): False
small whole numbers  : True


Rule: **keep money in `float64`** (or better, store integer cents). Use `float32` for ML **features** at the very end (section 14), when a tiny rounding difference does not matter.

## 4.3b Booleans, dates, categories

- `bool` columns are already 1 byte. Make sure flags are not stored as `int64` or text (`"Yes"/"No"`).
- Dates should use a pandas datetime-like dtype
  (such as `datetime64` or a timezone-aware datetime dtype), not plain text.
- Text with **few distinct values** should be `category` (next section).

## 4.4 A reusable `optimize()` function

In [14]:
def optimize(df, cat_threshold=0.5):
    """Shrink a DataFrame without losing information.
    - integers   -> smallest integer type that fits (safe)
    - floats     -> float32 ONLY if every value survives the conversion exactly (checked)
    - text       -> category if distinct values are less than `cat_threshold` of the rows
    """
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_datetime64_any_dtype(s):
            continue
        if pd.api.types.is_integer_dtype(s):
            out[col] = pd.to_numeric(s, downcast="integer")
        elif pd.api.types.is_float_dtype(s):
            as32 = s.astype("float32")
            if (as32.astype("float64").to_numpy() == s.to_numpy())[s.notna().to_numpy()].all():      # exact round trip
                out[col] = as32
        elif pd.api.types.is_string_dtype(s) or s.dtype == object:
            if s.nunique(dropna=True) / max(len(s), 1) < cat_threshold:
                out[col] = s.astype("category")
    return out


tx_small = optimize(tx)

report = pd.DataFrame({
    "before_dtype": tx.dtypes.astype(str),
    "after_dtype": tx_small.dtypes.astype(str),
    "before_MB": (tx.memory_usage(deep=True).drop("Index") / 1024 ** 2).round(2),
    "after_MB": (tx_small.memory_usage(deep=True).drop("Index") / 1024 ** 2).round(2),
})
print(report)
print()
print(f"TOTAL: {mb(tx)} MB -> {mb(tx_small)} MB  ({mb(tx) / mb(tx_small):.1f}x smaller)")

               before_dtype     after_dtype  before_MB  after_MB
tx_id                 int64           int32       3.81      1.91
customer_id           int64           int16       3.81      0.95
city                    str        category       6.39      0.48
category                str        category       6.91      0.48
amount              float64         float64       3.81      3.81
quantity              int64            int8       3.81      0.48
is_online              bool            bool       0.48      0.48
date         datetime64[us]  datetime64[us]       3.81      3.81

TOTAL: 32.86 MB -> 12.4 MB  (2.6x smaller)


In [15]:
# Prove that no information was lost: convert back to the ORIGINAL dtype and compare the real values
for col in tx.columns:
    pd.testing.assert_series_equal(tx[col], tx_small[col].astype(tx[col].dtype))
print("all values identical (amount stayed float64 because float32 would have changed it)")

all values identical (amount stayed float64 because float32 would have changed it)


**Important:** do the optimization **once, right after loading**, then work with the small version. Even better, tell `read_csv` the dtypes from the start (section 9), so the big version never exists in memory.

# 5. `category` in depth

A `category` column stores the **distinct values once** and every row as a small integer **code** that points to them.

```
category  : categories = ["Delhi", "Indore"],  codes = [1, 0, 1, 1, ...]
           (each row stores a compact integer code)
```

In [16]:
city = tx_small["city"]
print(city.dtype)
print("categories:", city.cat.categories.tolist())
print("first 6 codes:", city.cat.codes.head(6).tolist())
print("memory: text", round(tx["city"].memory_usage(deep=True) / 1024 ** 2, 2), "MB | category", round(city.memory_usage(deep=True) / 1024 ** 2, 2), "MB")

category
categories: ['Bhopal', 'Delhi', 'Indore', 'Mumbai', 'Pune']
first 6 codes: [3, 3, 0, 4, 0, 3]
memory: text 6.39 MB | category 0.48 MB


## When to use it (and when not)

| Use `category` | Do **not** use `category` |
|---|---|
| Few distinct values, many rows (city, status, plan, country) | Almost every value is unique (names, emails, IDs as text) |
| Group keys and merge keys used again and again | Column that you often modify with new, unseen values |
| Ordered levels (`Low < Medium < High`) | Tiny tables (no gain) |

## Pitfalls

In [17]:
c = pd.Series(["Indore", "Delhi", "Indore"], dtype="category")

# 1. You cannot put a value that is not a known category
try:
    c[0] = "Mumbai"
except Exception as e:
    print(type(e).__name__, "->", str(e)[:70])

# fix: add the category first
c = c.cat.add_categories(["Mumbai"])
c[0] = "Mumbai"
print(c.tolist())

TypeError -> Cannot setitem on a Categorical with a new category (Mumbai), set the 
['Mumbai', 'Delhi', 'Indore']


In [18]:
# 2. Combining columns with different categories falls back to plain text
a = pd.Series(["x", "y"], dtype="category")
b = pd.Series(["y", "z"], dtype="category")
print(pd.concat([a, b]).dtype)                    # no longer category

# fix: use the same categories for both
from pandas.api.types import union_categoricals
print(union_categoricals([a, b]).categories.tolist())

str
['x', 'y', 'z']


- Unused categories stay until you remove them: `.cat.remove_unused_categories()`.
- Sorting a category column follows the **category order** (not alphabetical) when the category is `ordered=True`.
- In pandas 3 `groupby` shows only categories that really occur (`observed=True`).
- For ML: `.cat.codes` gives the integer code (careful: a code is **not** a meaningful number, see chapter 06 on encoding).

# 6. Speed: vectorization (measured)

Same calculation, different styles. The task: `total = amount * quantity`, plus a discount rule. Test size: 100,000 rows.

In [19]:
sample = tx_small.head(100_000).copy()
sample["amount"] = tx["amount"].head(100_000)         # keep exact amounts

def loop_version():
    out = []
    for i in range(len(sample)):
        out.append(sample["amount"].iloc[i] * sample["quantity"].iloc[i])
    return out

def iterrows_version():
    return [row["amount"] * row["quantity"] for _, row in sample.head(20_000).iterrows()]      # only 20k rows: too slow otherwise

def apply_version():
    return sample.apply(lambda r: r["amount"] * r["quantity"], axis=1)

def zip_version():
    return [a * q for a, q in zip(sample["amount"], sample["quantity"])]

def vector_version():
    return sample["amount"] * sample["quantity"]

t = {
    "iloc loop (100k rows)":       bench(loop_version, repeat=1),
    "iterrows (only 20k rows!)":   bench(iterrows_version, repeat=1),
    "apply(axis=1) (100k rows)":   bench(apply_version, repeat=1),
    "zip list (100k rows)":        bench(zip_version),
    "vectorized (100k rows)":      bench(vector_version),
}

speed = pd.DataFrame({"seconds": t})
speed["x_slower_than_vectorized"] = (speed["seconds"] / speed.loc["vectorized (100k rows)", "seconds"]).round(0)
speed

,seconds,x_slower_than_vectorized
iloc loop (100k rows),2.018556,21036.0
iterrows (only 20k rows!),0.240383,2505.0
apply(axis=1) (100k rows),0.310601,3237.0
zip list (100k rows),0.012860,134.0
vectorized (100k rows),0.000096,1.0


Even on **5 times fewer rows**, `iterrows` is far slower than vectorized code on all rows. Vectorized code runs in compiled C loops over contiguous memory, row loops run Python bytecode with pandas overhead on **every** row.

## Conditional logic without loops

In [20]:
amt = sample["amount"]
qty = sample["quantity"]

def with_apply():
    return sample.apply(lambda r: r["amount"] * 0.9 if r["quantity"] >= 5 else r["amount"], axis=1)

def with_where():
    return np.where(qty >= 5, amt * 0.9, amt)

def with_select():
    return np.select([qty >= 8, qty >= 5], [amt * 0.8, amt * 0.9], default=amt)

print(f"apply(axis=1): {bench(with_apply, 1):.3f}s | np.where: {bench(with_where) * 1000:.1f}ms | np.select: {bench(with_select) * 1000:.1f}ms")
print("same result:", np.allclose(with_apply().to_numpy(), with_where()))

apply(axis=1): 0.326s | np.where: 0.1ms | np.select: 1.0ms
same result: True


## Text and `isin`

In [21]:
cities = tx["city"].head(200_000)
wanted = ["Indore", "Pune"]

def loop_in():
    return [c in wanted for c in cities]

def vector_isin():
    return cities.isin(wanted)

print(f"python loop: {bench(loop_in) * 1000:.1f}ms | isin: {bench(vector_isin) * 1000:.1f}ms")

python loop: 87.9ms | isin: 2.7ms


In [22]:
# .str methods are faster than apply with Python string functions
txt = pd.Series(["  Indore ", " BHOPAL", "pune  "] * 100_000)

print(f".str chain: {bench(lambda: txt.str.strip().str.title()) * 1000:.0f}ms | apply: {bench(lambda: txt.apply(lambda x: x.strip().title())) * 1000:.0f}ms")

.str chain: 8ms | apply: 48ms


## Never grow a DataFrame inside a loop

In [23]:
rows = [{"id": i, "value": i * 2} for i in range(1500)]

def grow_in_loop():
    df = pd.DataFrame(columns=["id", "value"])
    for r in rows:
        df = pd.concat([df, pd.DataFrame([r])], ignore_index=True)       # copies the whole table every time
    return df

def build_once():
    return pd.DataFrame(rows)                                             # collect rows in a list, build once

print(f"concat in a loop (1,500 rows): {bench(grow_in_loop, 1):.2f}s | build once: {bench(build_once) * 1000:.1f}ms")

concat in a loop (1,500 rows): 0.15s | build once: 0.5ms


**Order of preference** (fastest to slowest):

1. Built-in vectorized operations (`a * b`, `str`, `dt`, `isin`, `clip`, `rank`, `cumsum`)
2. `np.where` / `np.select`
3. `map` with a dict, `groupby` built-ins
4. List comprehension with `zip` (when you truly need Python logic)
5. `Series.apply`
6. `DataFrame.apply(axis=1)`
7. `iterrows`, loops with `iloc`, growing a DataFrame in a loop

# 7. Filtering, `query`, `eval`, and doing less work

In [24]:
big = tx_small
mask_time = bench(lambda: big[(big["amount"] > 500) & (big["category"] == "Electronics") & big["is_online"]])
query_time = bench(lambda: big.query("amount > 500 and category == 'Electronics' and is_online"))

print(f"boolean mask: {mask_time * 1000:.1f}ms | query: {query_time * 1000:.1f}ms")
print("same rows:", big[(big["amount"] > 500) & (big["category"] == "Electronics") & big["is_online"]].equals(
    big.query("amount > 500 and category == 'Electronics' and is_online")))

boolean mask: 0.9ms | query: 7.6ms
same rows: True


For normal data sizes, boolean masks are often just as fast or faster.
`query()` and `eval()` can become useful for large expressions, especially when
the `numexpr` engine is available. Do not assume they are automatically faster:
measure them on your real workload.

Use them mainly when the expression is clearer or when they reduce unnecessary
temporary arrays.

`eval` can create columns from an expression:

In [25]:
tmp = tx_small.head(5).copy()
tmp.eval("total = amount * quantity", inplace=True)         # (inplace is allowed for eval)
tmp[["amount", "quantity", "total"]]

,amount,quantity,total
0,108.66,2,217.32
1,104.86,1,104.86
2,129.72,4,518.88
3,173.40,4,693.60
4,90.53,5,452.65


## Do less work: filter early, select columns early

The cheapest row is the one you never process. Compare two ways to get "average amount of online Electronics purchases per city":

In [26]:
def late_filter():
    # does the heavy groupby on EVERYTHING, filters after
    g = tx_small.groupby(["city", "category", "is_online"], observed=True)["amount"].mean().reset_index()
    return g[(g["category"] == "Electronics") & g["is_online"]].groupby("city", observed=True)["amount"].mean()

def early_filter():
    # filters first, touches only 2 columns of the filtered rows
    sub = tx_small.loc[(tx_small["category"] == "Electronics") & tx_small["is_online"], ["city", "amount"]]
    return sub.groupby("city", observed=True)["amount"].mean()

print(f"filter late: {bench(late_filter) * 1000:.0f}ms | filter early: {bench(early_filter) * 1000:.0f}ms")

filter late: 20ms | filter early: 2ms


(The two give slightly different averages because the first one averages group averages. The point here is the amount of work, in a real task you would write the one that answers the question correctly.) The habits that matter:

- **Filter rows first**, then compute.
- **Select only the needed columns** before `merge`, `groupby` or `sort_values`.
- **Sort only when needed** (sorting is expensive). For the top N use `nlargest`.
- Use `drop_duplicates(subset=[...])` instead of sorting to find uniques.

# 8. GroupBy and merge performance

## 8.1 GroupBy

In [27]:
tx_text = tx.copy()                      # city and category as plain text
tx_cat = tx_small                        # city and category as category

def group_text():
    return tx_text.groupby(["city", "category"])["amount"].mean()

def group_cat():
    return tx_cat.groupby(["city", "category"], observed=True)["amount"].mean()

def group_cat_nosort():
    return tx_cat.groupby(["city", "category"], observed=True, sort=False)["amount"].mean()

def group_lambda():
    return tx_cat.groupby(["city", "category"], observed=True)["amount"].agg(lambda s: s.mean())

t = {"text keys": bench(group_text), "category keys": bench(group_cat),
     "category + sort=False": bench(group_cat_nosort), "lambda instead of 'mean'": bench(group_lambda)}
pd.DataFrame({"ms": {k: round(v * 1000, 1) for k, v in t.items()}})

,ms
text keys,29.6
category keys,24.8
category + sort=False,23.0
lambda instead of 'mean',26.9


- **Category keys** are usually faster than text keys.
- **`sort=False`** skips sorting the groups (it helps when there are many groups, with only 30 groups the saving is too small to see).
- **Built-in names** (`"mean"`, `"sum"`) beat `lambda` (chapter 10).
- Several aggregations in one `agg(...)` call are faster than several separate groupbys.

## 8.2 Merge

In [28]:
customers = pd.DataFrame({
    "customer_id": np.arange(1, 20_001),
    "segment": np.random.default_rng(1).choice(["A", "B", "C"], 20_000),
})

def merge_on_column():
    return tx_small[["customer_id", "amount"]].merge(customers, on="customer_id", how="left")

def merge_all_columns():
    return tx.merge(customers, on="customer_id", how="left")

cust_idx = customers.set_index("customer_id")
def join_on_index():
    return tx_small[["customer_id", "amount"]].join(cust_idx, on="customer_id")

def map_lookup():
    return tx_small["customer_id"].map(cust_idx["segment"])

r_small, r_all = merge_on_column(), merge_all_columns()
print(f"merge, only needed columns: {bench(merge_on_column) * 1000:.0f}ms | result {r_small.shape[1]} columns, {mb(r_small)} MB")
print(f"merge, ALL columns        : {bench(merge_all_columns) * 1000:.0f}ms | result {r_all.shape[1]} columns, {mb(r_all)} MB")
print(f"join on index             : {bench(join_on_index) * 1000:.0f}ms")
print(f"map lookup (one column)   : {bench(map_lookup) * 1000:.0f}ms")

merge, only needed columns: 19ms | result 3 columns, 9.12 MB
merge, ALL columns        : 22ms | result 9 columns, 37.21 MB
join on index             : 20ms
map lookup (one column)   : 9ms


- Merge **only the columns you need** (select before the merge). The time difference is small here (pandas 3 copies lazily), but the **result is much smaller in memory**, and the gap grows with wide tables.
- `map` with a lookup Series is the fastest way to add **one** column from a lookup table.
- Merge keys with the **same dtype** are faster, and avoid errors.
- Make sure the lookup key is **unique** (`validate="many_to_one"`), duplicates multiply rows (chapter 11).

# 9. File formats and reading less

## 9.1 Same data, different files

In [29]:
tx.to_csv(DATA / "tx.csv", index=False)
tx_small.to_parquet(DATA / "tx.parquet", index=False)

size = lambda p: round(p.stat().st_size / 1024 ** 2, 1)
print("CSV    :", size(DATA / "tx.csv"), "MB")
print("Parquet:", size(DATA / "tx.parquet"), "MB")

CSV    : 24.5 MB
Parquet: 6.0 MB


In [30]:
def read_default():
    return pd.read_csv(DATA / "tx.csv")

def read_tuned():
    return pd.read_csv(
        DATA / "tx.csv",
        usecols=["customer_id", "city", "amount", "quantity"],
        dtype={"customer_id": "int32", "city": "category", "quantity": "int8"},
    )

def read_parquet_all():
    return pd.read_parquet(DATA / "tx.parquet")

def read_parquet_cols():
    return pd.read_parquet(DATA / "tx.parquet", columns=["customer_id", "city", "amount", "quantity"])

times = {
    "CSV, default":                 bench(read_default, 1),
    "CSV, usecols + dtype":         bench(read_tuned, 1),
    "Parquet, all columns":         bench(read_parquet_all, 1),
    "Parquet, only 4 columns":      bench(read_parquet_cols, 1),
}

try:
    times["CSV, engine='pyarrow'"] = bench(lambda: pd.read_csv(DATA / "tx.csv", engine="pyarrow"), 1)
except ImportError:
    print("pyarrow not installed: pip install pyarrow")

pd.DataFrame({"seconds": {k: round(v, 2) for k, v in times.items()}}).sort_values("seconds")

,seconds
"Parquet, only 4 columns",0.01
"CSV, engine='pyarrow'",0.04
"CSV, usecols + dtype",0.14
"Parquet, all columns",0.24
"CSV, default",0.29


In [31]:
tuned = read_tuned()
print("memory of the default read :", mb(read_default()), "MB")
print("memory of the tuned read   :", mb(tuned), "MB")

memory of the default read : 37.62 MB
memory of the tuned read   : 6.68 MB


What to take from this:

| Trick | Effect |
|---|---|
| `usecols=[...]` | reads fewer columns: less time **and** memory |
| `dtype={...}` | right dtype from the start (no big temporary) |
| `nrows=...` | develop and debug on a small sample first |
| `engine="pyarrow"` | multi-threaded CSV parser, often much faster |
| **Parquet** | smaller file, keeps dtypes (including `category`), can read **only some columns** (the biggest win: see the timing of the 4-column read) |

**Habit:** convert big CSVs to Parquet once (`to_parquet`), then always read the Parquet.

# 10. Processing files bigger than memory (chunks)

When even the tuned read does not fit, read **in chunks**, compute a **small partial result** per chunk, and **combine** the partial results.

The partial results must be **combinable**:

| Statistic | Keep per chunk | Combine |
|---|---|---|
| sum, count, min, max | the value | sum / sum / min / max |
| **mean** | **sum and count** | total sum / total count (never average the averages) |
| nunique, median | hard (need all values or sketches) | use a database, or approximate methods |

In [32]:
# Per-customer total, count and max amount, computed in chunks of 100,000 rows
partials = []
for chunk in pd.read_csv(
    DATA / "tx.csv", usecols=["customer_id", "amount"], dtype={"customer_id": "int32"}, chunksize=100_000
):
    partials.append(chunk.groupby("customer_id")["amount"].agg(["sum", "count", "max"]))

combined = pd.concat(partials).groupby(level=0).agg({"sum": "sum", "count": "sum", "max": "max"})
combined["mean"] = combined["sum"] / combined["count"]
print(len(partials), "chunks ->", len(combined), "customers")
combined.head()

5 chunks -> 20000 customers


,sum,count,max,mean
customer_id,,,,
1,8414.07,38,1851.53,221.422895
2,2879.00,20,830.56,143.950000
3,6268.40,32,575.34,195.887500
4,5636.34,27,755.58,208.753333
5,4805.02,12,2163.37,400.418333


In [33]:
# Check against the full computation
direct = tx.groupby("customer_id")["amount"].agg(["sum", "count", "max", "mean"])
print("sums match :", np.allclose(combined["sum"], direct["sum"]))
print("means match:", np.allclose(combined["mean"], direct["mean"]))

sums match : True
means match: True


Only **one chunk at a time** is in memory (100,000 rows instead of everything), plus the small partial tables. The same code works for a file of 500 million rows.

# 11. Copies, Copy-on-Write and freeing memory

## Copy-on-Write (pandas 3)

In pandas 3, Copy-on-Write is enabled by default.
A selection or method result behaves independently from the original, while
the underlying data may be copied only when a modification requires it.

This avoids many unnecessary defensive `.copy()` calls.
Use an explicit `.copy()` when you intentionally need a fully independent copy,
for example before an operation where you specifically want a separate materialized object.

In [34]:
t_select = bench(lambda: tx_small[["amount", "quantity"]], repeat=5)
t_copy = bench(lambda: tx_small[["amount", "quantity"]].copy(), repeat=5)
print(f"select 2 columns: {t_select * 1000:.2f}ms | select + copy: {t_copy * 1000:.2f}ms")

select 2 columns: 0.19ms | select + copy: 0.51ms


Still true:
- **`df.copy()`** is only needed when you want a separate object **and** will modify it in a way that must not affect the original. Do not sprinkle it everywhere.
- Do not assume `inplace=True` is faster or uses less memory. Prefer clear reassignment:
  `df = df.method(...)`

## Freeing memory

In [35]:
big_tmp = tx.copy()
print("with temp copy:", mb(big_tmp), "MB")

del big_tmp                  # remove the name, then ask Python to clean up
gc.collect()
print("deleted")

with temp copy: 32.86 MB
deleted


Good habits for long notebooks:
- Delete large intermediate tables you no longer need (`del name`).
- Re-assign the same name in a pipeline (`df = df.assign(...)`), the old object is freed automatically.
- Restart the kernel from time to time: notebooks keep everything you ever created.
- Keep **one small, optimized version** of the data (`tx_small`) and work with it.

# 12. Method chaining and clean pipelines

Fast code is useless if nobody (including you in 3 months) can read it. A chain reads top to bottom as a story and creates **no scattered temporary variables**.

In [36]:
monthly_online_revenue = (
    tx_small
    .loc[tx_small["is_online"]]                                          # only online purchases
    .assign(
        revenue=lambda d: d["amount"] * d["quantity"],                   # new column
        month=lambda d: d["date"].dt.to_period("M"),
    )
    .groupby(["month", "category"], observed=True)["revenue"]
    .sum()
    .unstack("category")
    .round(0)
)
monthly_online_revenue.head(3)

category,Electronics,Fashion,Food,Grocery,Home,Travel
month,,,,,,
2025-01,3562689.0,3633735.0,3667019.0,3796966.0,3621007.0,3473422.0
2025-02,3084527.0,3163459.0,3087875.0,3219562.0,3026715.0,3199588.0
2025-03,3554471.0,3357220.0,3473527.0,3500190.0,3644235.0,3426920.0


In [37]:
# Debug a chain: peek into the middle without breaking it
def peek(d, label=""):
    print(f"[{label}] shape={d.shape}")
    return d                                                              # always return the data

check = (
    tx_small
    .pipe(peek, "start")
    .query("amount > 500")
    .pipe(peek, "after filter")
    .groupby("city", observed=True)["amount"].mean()
)
check.round(1)

[start] shape=(500000, 8)
[after filter] shape=(56342, 8)


city
Bhopal    906.5
Delhi     899.3
Indore    909.0
Mumbai    898.3
Pune      907.1
Name: amount, dtype: float64

Best practices for readable and fast pipelines:

1. **One function per step** (or one chain) that takes a DataFrame and returns a DataFrame (`pipe`).
2. **Name things** clearly. `df2`, `df3`, `tmp` are a smell.
3. Put **settings** (paths, column names, thresholds) at the top as constants.
4. Add **checks** (`assert len(result) == len(left)`, `validate=...`), chapters 08 and 11.
5. Keep **raw data untouched**, write cleaned data to a **new** file (Parquet).
6. **Seed** every random operation for reproducibility.

# 13. When pandas is not enough: DuckDB and Polars

pandas does not automatically parallelize every DataFrame operation across
all CPU cores, and its normal DataFrame workflow keeps the working data in RAM.

Signs you may need another tool:
- data is larger than available RAM
- important operations are consistently too slow
- you need stronger parallel or lazy execution
- SQL-style analytics or direct file querying is a better fit

| Tool | What it is | Good for |
|---|---|---|
| **DuckDB** | SQL engine that runs inside Python, can query DataFrames and Parquet/CSV files **directly**, bigger than memory | analytics with SQL, big files, joins and group by |
| **Polars** | DataFrame library written in Rust, multi-threaded, lazy execution | fast analytical pipelines, especially with Parquet |
| **Dask / Spark** | distributed DataFrames | many machines, terabytes |
| **Database (PostgreSQL)** | filter and aggregate **inside** the database | data already lives there (your `24_postgre_sql` folder) |

They can be used **together** with pandas: do the heavy work in DuckDB/Polars, then bring the **small result** into pandas with `.df()` / `.to_pandas()`.

Install: `pip install duckdb polars` (the cells below skip politely if they are missing).

In [38]:
query = "SELECT city, category, AVG(amount) AS avg_amount, COUNT(*) AS n FROM tx_small GROUP BY city, category ORDER BY city, category"

try:
    import duckdb

    start = time.perf_counter()
    duck = duckdb.sql(query).df()                        # DuckDB reads the pandas DataFrame by its variable name
    t_duck = time.perf_counter() - start

    start = time.perf_counter()
    pand = (tx_small.groupby(["city", "category"], observed=True)["amount"].agg(avg_amount="mean", n="count")
            .reset_index().sort_values(["city", "category"]).reset_index(drop=True))
    t_pandas = time.perf_counter() - start

    print(f"pandas: {t_pandas * 1000:.0f}ms | DuckDB: {t_duck * 1000:.0f}ms")
    print("same numbers:", np.allclose(duck["avg_amount"], pand["avg_amount"]))
    print(duck.head(3))
except ImportError:
    print("DuckDB is not installed: pip install duckdb")

DuckDB is not installed: pip install duckdb


In [39]:
# DuckDB can query FILES directly, without loading them into pandas first
try:
    import duckdb
    print(duckdb.sql("SELECT city, SUM(amount) AS total FROM read_parquet('data/tx.parquet') GROUP BY city ORDER BY total DESC").df())
except ImportError:
    print("DuckDB is not installed: pip install duckdb")

DuckDB is not installed: pip install duckdb


In [40]:
try:
    import polars as pl

    def polars_query():
        return (
            pl.scan_parquet(DATA / "tx.parquet")                       # lazy: nothing is read yet
              .filter(pl.col("is_online"))
              .group_by("city")
              .agg(pl.col("amount").mean().alias("avg_amount"), pl.len().alias("n"))
              .sort("city")
              .collect()                                               # run the optimized plan now
        )

    polars_query()                                                     # warm-up run (first call includes one-time setup)
    print(f"polars (second run): {bench(polars_query) * 1000:.0f}ms")
    print(polars_query().to_pandas().round(1))
except ImportError:
    print("Polars is not installed: pip install polars")

Polars is not installed: pip install polars


Timings depend heavily on the machine, data size, query, and workload.
DuckDB and Polars can outperform pandas on large analytical workloads because
they provide strong parallel execution and query optimization.

Polars' `scan_parquet()` builds a lazy query plan, and DuckDB can query Parquet
files directly without first loading the complete file into pandas.

At this size the differences may be small. The right takeaway is not "never use
pandas" but: know the alternatives and choose the tool that fits the workload.

# 14. Mini AI/ML example: an efficient feature pipeline

Goal: build a **customer feature table** from the 500,000 transactions, ready for a model, with a small memory footprint.

Steps: read only what we need with the right dtypes, aggregate per customer (RFM-style, chapter 10), add features, shrink dtypes, make a time-safe label, and hand over `float32` arrays.

In [41]:
cutoff = pd.Timestamp("2025-10-01")

# 1. Read once, only needed columns, right dtypes (Parquet keeps the dtypes already)
cols = ["customer_id", "category", "amount", "quantity", "is_online", "date"]
raw = pd.read_parquet(DATA / "tx.parquet", columns=cols)

history = raw[raw["date"] < cutoff]                           # what is known at prediction time
future = raw[(raw["date"] >= cutoff) & (raw["date"] < cutoff + pd.Timedelta(days=7))]       # the next 7 days

# 2. One groupby with several named aggregations (fast, readable)
features = history.groupby("customer_id", observed=True).agg(
    frequency=("amount", "count"),
    monetary=("amount", "sum"),
    avg_amount=("amount", "mean"),
    max_amount=("amount", "max"),
    avg_quantity=("quantity", "mean"),
    online_share=("is_online", "mean"),
    last_date=("date", "max"),
    n_categories=("category", "nunique"),
)
features["recency_days"] = (cutoff - features["last_date"]).dt.days
features = features.drop(columns="last_date")

# 3. Time-safe label: does the customer buy within 7 days after the cutoff?
features["buys_next_7d"] = features.index.isin(future["customer_id"].unique()).astype("int8")

print(features.shape, "| memory:", mb(features), "MB")
features.head(3).round(2)

(20000, 9) | memory: 1.28 MB


,frequency,monetary,avg_amount,max_amount,avg_quantity,online_share,n_categories,recency_days,buys_next_7d
customer_id,,,,,,,,,
1,25,4703.32,188.13,488.53,4.40,0.40,6,11,0
2,15,2623.19,174.88,830.56,5.13,0.33,6,34,0
3,23,4858.89,211.26,575.34,4.83,0.43,6,9,1


In [42]:
# 4. Category mix per customer (a wide block of features), stored compactly
mix = (
    history.groupby(
        ["customer_id", "category"],
        observed=True
    )["amount"]
    .sum()
    .unstack(fill_value=0)
    .add_prefix("spend_")
)

table = features.join(mix, how="left")
table = optimize(table)                                       # shrink what can be shrunk safely

print("final table:", table.shape, "| memory:", mb(table), "MB")
print(table.dtypes.astype(str).value_counts().to_dict())

final table: (20000, 15) | memory: 1.81 MB
{'float64': 11, 'int8': 3, 'int16': 1}


#### 5. Prepare model inputs

The feature table is already built using information available before the cutoff,
and the label is defined from the future window.

For an actual ML experiment, create train/validation/test splits separately.
For time-dependent problems, those splits should respect time.

In [43]:
# 5. Prepare model inputs and compact arrays
X = table.drop(columns="buys_next_7d")
y = table["buys_next_7d"]

X32 = X.to_numpy(dtype="float32")
X64 = X.to_numpy(dtype="float64")
print("X float64:", round(X64.nbytes / 1024 ** 2, 2), "MB | X float32:", round(X32.nbytes / 1024 ** 2, 2), "MB")
print("label balance:", y.mean().round(3))

X float64: 2.14 MB | X float32: 1.07 MB
label balance: 0.384


Summary of the pipeline's efficiency habits:
1. Read **only needed columns** from **Parquet**
2. **One** `groupby` with several named aggregations instead of many small ones
3. **Built-in** aggregations only (no lambdas)
4. Compact dtypes (`optimize`, `float32` for model inputs)
5. **Time-safe**: features before the cutoff, label after it
6. Everything is a small, readable, repeatable function (put it in `project/` later)

# 15. Pandas 3.x notes

- **Copy-on-Write** is the default: selections and method results behave independently,
  while physical copying is delayed until a modification requires it. Do not assume
  `inplace=True` provides a memory or speed benefit.
- Text columns use the **`str` dtype** (backed by Arrow when `pyarrow` is installed), which is already much more compact and faster than the old `object`. `category` still wins for low-cardinality columns.
- `read_csv(engine="pyarrow")` and `read_parquet` give fast, multi-threaded reading when `pyarrow` is installed. `dtype_backend="pyarrow"` can be used to get Arrow-backed numeric dtypes too.
- `groupby` uses `observed=True` by default for categoricals, which avoids huge result tables with empty combinations.
- Several deprecated slow paths were removed (`applymap`, `append`, `method=` in `fillna`).
- Downcasting manually with `astype("int8")` can **silently overflow**. Use `pd.to_numeric(..., downcast=...)`.

# 16. Common mistakes

| Mistake | Fix |
|---|---|
| Optimizing without measuring | `%timeit`, `bench`, `memory_usage(deep=True)` first |
| `memory_usage()` without `deep=True` for text columns | `deep=True` |
| `astype("int8")` on unchecked data (silent overflow) | `pd.to_numeric(downcast="integer")` |
| `float32` for money, or trusting `downcast="float"` (it checks only the range, not the precision) | Keep `float64` or use integer cents; check the round trip before converting |
| `category` for a column with almost unique values | Plain text/`str` |
| Reading the whole CSV when 3 columns are needed | `usecols`, Parquet `columns=` |
| Letting `read_csv` guess dtypes on a huge file | Pass `dtype=` |
| `iterrows`, `apply(axis=1)` on big tables | Vectorize, `np.where`, `np.select`, `zip` |
| Growing a DataFrame in a loop with `concat` | Collect rows in a list, build once |
| Merging with all columns, then dropping most | Select columns before the merge |
| Averaging chunk averages | Keep sum and count per chunk |
| `lambda` in `agg` for sum/mean/max | Built-in string names |
| `.copy()` and `inplace=True` everywhere | Use `.copy()` when you intentionally need an independent copy; do not assume `inplace=True` improves performance |
| Keeping many big intermediate DataFrames alive | `del`, re-assign, restart the kernel |
| Fighting pandas for a 100 GB dataset | DuckDB, Polars, database, Spark |
| Unreadable 30-line mutation scripts | Method chains, `pipe`, small functions |
| Random operations without a seed | `np.random.default_rng(seed)` |

---
# 17. Interview questions

**Q1. A CSV does not fit in memory. What do you do?**
Read only the needed columns (`usecols`) with compact dtypes (`dtype=`), process in `chunksize` pieces with combinable partial results (sum and count), convert to Parquet, or use DuckDB/Polars/a database.

**Q2. How do you reduce the memory of a DataFrame?**
Downcast integers safely (`pd.to_numeric(downcast="integer")`), floats only when lossless, convert low-cardinality text to `category`, make sure dates are `datetime64` and flags are `bool`, drop unused columns, and load fewer columns from the start.

**Q3. What is the `category` dtype and when is it useful?**
It stores distinct values once and each row as a small integer code. It saves memory and speeds up groupby/merge for columns with few distinct values repeated many times. It is a bad fit for nearly unique columns.

**Q4. Why is vectorized code faster than `apply` or loops?**
Vectorized operations run in compiled code over contiguous arrays. `apply`/loops call a Python function per row and pay interpreter and pandas overhead each time.

**Q5. Order these by speed: `iterrows`, `apply(axis=1)`, vectorized, `zip` loop, `np.where`.**
Fastest: vectorized and `np.where`; then `zip` loop; then `apply(axis=1)`; slowest: `iterrows`.

**Q6. What is the risk of `astype("int8")`?**
Values outside the range silently wrap around (200 becomes -56). Use `pd.to_numeric(downcast="integer")`, which picks an integer type that fits. For floats, `downcast="float"` checks only the range, so verify precision yourself.

**Q7. CSV vs Parquet?**
Parquet is binary and columnar: smaller, faster, keeps dtypes, and can read only some columns. CSV is text: bigger, slow, dtypes are guessed again every time, but human readable and universal.

**Q8. Why is `df.merge` slow and how do you speed it up?**
Too many columns, mismatched key dtypes, duplicate keys (row multiplication), or many-to-many joins. Select only needed columns first, use the same dtype (or category) for keys, ensure unique lookup keys, or use `map` for one column.

**Q9. How do you compute a mean over chunks?**
Keep the sum and the count of each chunk and divide the totals at the end. Averaging chunk means is wrong when the chunks have different sizes.

**Q10.** Starting from `pdata`, write one method chain that:
1. adds `total = price * items`
2. keeps rows with `paid`
3. averages `total` per `device`
4. sorts from high to low
Do not add an unnecessary `.copy()`; pandas 3 uses Copy-on-Write.

**Q11. When would you move from pandas to DuckDB or Polars?**
When data is larger than RAM, important operations are consistently too slow,
or you need stronger parallelism and lazy optimization. Heavy lifting goes into
DuckDB/Polars and the small result comes back to pandas.

**Q12. How do you profile slow pandas code?**
`%timeit` / `time.perf_counter` for steps, `df.memory_usage(deep=True)` for memory, and `cProfile` or `line_profiler` to find the slow line. Always measure before and after a change.

---
# 18. Practice questions

Run this cell first (a fresh 200,000-row table for the exercises, and a CSV file).

In [44]:
rng = np.random.default_rng(7)
m = 200_000

pdata = pd.DataFrame({
    "order_id": np.arange(m),
    "user_id":  rng.integers(1, 5001, m),
    "country":  rng.choice(["India", "USA", "UK", "Germany", "Japan"], m),
    "device":   rng.choice(["mobile", "desktop", "tablet"], m, p=[0.6, 0.3, 0.1]),
    "price":    rng.lognormal(4, 0.8, m).round(2),
    "items":    rng.integers(1, 15, m),
    "paid":     rng.random(m) < 0.9,
})
pdata.to_csv(DATA / "practice.csv", index=False)
pdata.head(3)

,order_id,user_id,country,device,price,items,paid
0,0,4725,India,mobile,38.66,3,True
1,1,3126,Germany,desktop,133.70,9,True
2,2,3421,UK,mobile,23.10,8,True


**Q1.** Show the memory (MB) of every column and the total.
**Q2.** Convert `country` and `device` to `category`. How much memory is saved (percent)?
**Q3.** Downcast `items` and `user_id` safely. What dtypes did you get? Then show what `astype("int8")` would do to `user_id` (the dangerous way).
**Q4.** Compute `total = price * items` (a) with `apply(axis=1)` on the first 20,000 rows and (b) vectorized. How many times slower is (a)?
**Q5.** Select paid orders from `mobile` with `price > 100`, with a boolean mask and with `query`. Are the results equal?
**Q6.** Read `practice.csv` again but only `country`, `price`, `items`, with compact dtypes. Compare the memory with a default full read.
**Q7.** Read the CSV in chunks of 50,000 rows and compute the total `price` per `country` (chunk by chunk). Compare with the direct result.
**Q8.** Measure `groupby("country")["price"].mean()` with `country` as text and as category. Which is faster?
**Q9.** Rewrite this slow pattern with the list approach: building a DataFrame of 2,000 rows (`id`, `square`) by concatenating one row at a time.
**Q10.** Starting from `pdata`, write one method chain that:
1. adds `total = price * items`
2. keeps rows with `paid`
3. averages `total` per `device`
4. sorts from high to low
Do not add an unnecessary `.copy()`; pandas 3 uses Copy-on-Write.
**Q11.** Save `pdata` as Parquet (with `country` and `device` as category) and compare the file size with the CSV. Read only 2 columns from the Parquet.
**Q12.** Write your own `shrink(df)` that downcasts numeric columns safely (floats only when lossless) and converts text columns with fewer than 50% distinct values to `category`. Use it on `pdata` and verify that the values are unchanged.

### Solutions

In [45]:
# Q1
mem = pdata.memory_usage(deep=True).drop("Index")
print((mem / 1024 ** 2).round(2))
print("total MB:", round(mem.sum() / 1024 ** 2, 2))

order_id    1.53
user_id     1.53
country     2.37
device      2.73
price       1.53
items       1.53
paid        0.19
dtype: float64
total MB: 11.39


In [46]:
# Q2
p2 = pdata.astype({"country": "category", "device": "category"})
saved = (1 - mb(p2) / mb(pdata)) * 100
print(f"{mb(pdata)} MB -> {mb(p2)} MB  ({saved:.0f}% saved)")

11.39 MB -> 6.68 MB  (41% saved)


In [47]:
# Q3
print(pd.to_numeric(pdata["items"], downcast="integer").dtype, pd.to_numeric(pdata["user_id"], downcast="integer").dtype)
print("dangerous int8:", pdata["user_id"].head(5).tolist(), "->", pdata["user_id"].head(5).astype("int8").tolist())
# user_id goes up to 5000, which does not fit in int8 (max 127): the values are corrupted

int8 int16
dangerous int8: [4725, 3126, 3421, 4487, 2892] -> [117, 54, 93, -121, 76]


In [48]:
# Q4
head = pdata.head(20_000)
t_apply = bench(lambda: head.apply(lambda r: r["price"] * r["items"], axis=1), 1)
t_vec = bench(lambda: head["price"] * head["items"])
print(f"apply is about {t_apply / t_vec:,.0f}x slower")

apply is about 1,270x slower


In [49]:
# Q5
a = pdata[pdata["paid"] & (pdata["device"] == "mobile") & (pdata["price"] > 100)]
b = pdata.query("paid and device == 'mobile' and price > 100")
print(len(a), a.equals(b))

24386 True


In [50]:
# Q6
slim = pd.read_csv(DATA / "practice.csv", usecols=["country", "price", "items"],
                   dtype={"country": "category", "items": "int8"})
full = pd.read_csv(DATA / "practice.csv")
print("full read:", mb(full), "MB | slim read:", mb(slim), "MB")

full read: 11.39 MB | slim read: 1.91 MB


In [51]:
# Q7
parts = []
for chunk in pd.read_csv(DATA / "practice.csv", usecols=["country", "price"], chunksize=50_000):
    parts.append(chunk.groupby("country")["price"].sum())

by_chunks = pd.concat(parts).groupby(level=0).sum()
direct = pdata.groupby("country")["price"].sum()
print(np.allclose(by_chunks.sort_index(), direct.sort_index()))
by_chunks.round(0)

True


country
Germany    3048135.0
India      3005488.0
Japan      2988780.0
UK         3008078.0
USA        3044203.0
Name: price, dtype: float64

In [52]:
# Q8
text_df = pdata[["country", "price"]]
cat_df = pdata[["country", "price"]].astype({"country": "category"})
t_text = bench(lambda: text_df.groupby("country")["price"].mean())
t_cat = bench(lambda: cat_df.groupby("country", observed=True)["price"].mean())
print(f"text: {t_text * 1000:.1f}ms | category: {t_cat * 1000:.1f}ms")

text: 5.4ms | category: 2.5ms


In [53]:
# Q9
def slow():
    df = pd.DataFrame(columns=["id", "square"])
    for i in range(2000):
        df = pd.concat([df, pd.DataFrame([{"id": i, "square": i ** 2}])], ignore_index=True)
    return df

def fast():
    return pd.DataFrame([{"id": i, "square": i ** 2} for i in range(2000)])

print(f"slow: {bench(slow, 1):.2f}s | fast: {bench(fast) * 1000:.1f}ms | same: {slow().astype('int64').equals(fast())}")

slow: 0.22s | fast: 1.2ms | same: True


In [54]:
# Q10
result = (
    pdata
    .assign(total=lambda d: d["price"] * d["items"])
    .loc[lambda d: d["paid"]]
    .groupby("device")["total"].mean()
    .sort_values(ascending=False)
)
result.round(1)

device
desktop    572.1
mobile     563.6
tablet     558.2
Name: total, dtype: float64

In [55]:
# Q11
pq = pdata.astype({"country": "category", "device": "category"})
pq.to_parquet(DATA / "practice.parquet", index=False)
print("CSV:", round((DATA / "practice.csv").stat().st_size / 1024 ** 2, 1), "MB | Parquet:",
      round((DATA / "practice.parquet").stat().st_size / 1024 ** 2, 1), "MB")
pd.read_parquet(DATA / "practice.parquet", columns=["country", "price"]).head(3)

CSV: 7.2 MB | Parquet: 2.1 MB


,country,price
0,India,38.66
1,Germany,133.70
2,UK,23.10


In [56]:
# Q12
def shrink(df, threshold=0.5):
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_bool_dtype(s):
            continue
        if pd.api.types.is_integer_dtype(s):
            out[col] = pd.to_numeric(s, downcast="integer")
        elif pd.api.types.is_float_dtype(s):
            as32 = s.astype("float32")
            if (as32.astype("float64") == s).all():                  # only if lossless
                out[col] = as32
        elif pd.api.types.is_string_dtype(s) or s.dtype == object:
            if s.nunique() / len(s) < threshold:
                out[col] = s.astype("category")
    return out

small = shrink(pdata)
same = all(pdata[c].equals(small[c].astype(pdata[c].dtype)) for c in pdata.columns)       # real values, not text
print(f"{mb(pdata)} MB -> {mb(small)} MB | values unchanged: {same}")
print(small.dtypes.astype(str).to_dict())

11.39 MB -> 3.43 MB | values unchanged: True
{'order_id': 'int32', 'user_id': 'int16', 'country': 'category', 'device': 'category', 'price': 'float64', 'items': 'int8', 'paid': 'bool'}


---
# 19. Cheat sheet

```python
# MEASURE
%timeit expr          time.perf_counter()          df.memory_usage(deep=True)          df.info(memory_usage="deep")

# MEMORY
pd.to_numeric(s, downcast="integer")      # SAFE (never astype("int8") on unchecked data)
# floats: convert to float32 only after checking that (s.astype('float32').astype('float64') == s).all()
df["col"] = df["col"].astype("category")  # few distinct values, many rows
c.cat.add_categories([...])   c.cat.remove_unused_categories()
float64 for money, float32 for model inputs at the very end
del big_df; gc.collect()

# SPEED (fast -> slow)
vectorized ops > np.where / np.select > map(dict) > groupby built-ins > zip comprehension > Series.apply > apply(axis=1) > iterrows
rows = []; rows.append(...); df = pd.DataFrame(rows)          # never concat in a loop
df.query("a > 5 and b == 'x'")      # readable filtering; benchmark if speed matters
df.eval("c = a + b")                # expression evaluation; benchmark if speed matters
filter rows first, select columns first, sort only when needed

# GROUPBY / MERGE
groupby(..., observed=True, sort=False)["x"].agg(total="sum", avg="mean")     # built-ins, no lambda
left[["key", "x"]].merge(right[["key", "y"]], on="key", validate="many_to_one")
s.map(lookup_series)                                                           # one column from a lookup

# READ LESS
pd.read_csv(path, usecols=[...], dtype={...}, nrows=1000, engine="pyarrow")
df.to_parquet("x.parquet");   pd.read_parquet("x.parquet", columns=[...])
for chunk in pd.read_csv(path, chunksize=100_000): partials.append(chunk.groupby(k)[x].agg(["sum", "count"]))
total = pd.concat(partials).groupby(level=0).sum();   mean = total["sum"] / total["count"]

# CLEAN CODE
df.pipe(f).assign(new=lambda d: ...).query("...").groupby(...).agg(...)
constants at the top, asserts after merges, raw data untouched, seeds for randomness

# BIGGER THAN PANDAS
import duckdb;  duckdb.sql("SELECT ... FROM df_or_'file.parquet' GROUP BY ...").df()
import polars as pl;  pl.scan_parquet("x.parquet").filter(...).group_by(...).agg(...).collect().to_pandas()
```

**Next chapter:** `15_visualization_basics`: plotting directly from pandas (`df.plot`), Matplotlib essentials, Seaborn for statistical plots, and the plots every EDA should have (distributions, relationships, categories, time series, correlation heatmap, missing-value map).

After that: `16_project`, the final real-world AI/ML project that uses everything from chapters 01 to 15.